**Actividad Fundamental 7: Redes Neuronales Artificiales**

**Docente:** Ing. Daniel Isaias López Páez

**Hora de clase:** Jueves N1-N3

**Integrantes:**
* Pedro Daniel Carbajal Cruz - 2222798
* Daniel Rodríguez Arellano - 2041848
* José Adrián Hernández Hernández - 1891823
* Edgar Ivan Martinez Ramirez - 2120721
* Luis Angel Rodriguez Gutierrez - 2059357

**IMPORTAMOS LIBRERIAS**

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
from sklearn.model_selection import train_test_split
from google.colab import drive

print("Versión de TensorFlow:", tf.__version__)

**Carga del Conjunto de Datos (Dataset)**

In [ ]:
# Montar Google Drive para acceder a la carpeta del dataset en Colab
drive.mount('/content/drive')

# Ajusta esta ruta según la ubicación de tu carpeta 'dataset' en tu Google Drive
ruta_dataset = '/content/drive/MyDrive/dataset'
IMG_SIZE = 64  # Tamaño al que se redimensionarán las imágenes para la CNN

X_data = []
y_data = []

class_names = sorted(os.listdir(ruta_dataset))
class_names = [c for c in class_names if os.path.isdir(os.path.join(ruta_dataset, c))]
print(f"Clases de colores encontradas: {class_names}")
num_classes = len(class_names)

for idx, clase in enumerate(class_names):
    carpeta_clase = os.path.join(ruta_dataset, clase)
    imagenes_en_clase = 0
    
    for nombre_archivo in os.listdir(carpeta_clase):
        if nombre_archivo.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
            ruta_imagen = os.path.join(carpeta_clase, nombre_archivo)
            
            img = cv2.imread(ruta_imagen)
            if img is not None:
                img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
                
                X_data.append(img_rgb)
                y_data.append(idx)
                imagenes_en_clase += 1

    print(f"-> Categoría '{clase}': se leyeron {imagenes_en_clase} imágenes correctamente.")

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int32)

print(f"\n¡Total de imágenes cargadas: {len(X_data)}!")

**Procesamiento De Datos**

In [ ]:
# Normalizar los valores de los píxeles (0 a 1)
X_data /= 255.0

# Separación en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
train_data, test_data, train_labels, test_labels = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {train_data.shape}")
print(f"Datos de prueba: {test_data.shape}")
print("Preprocesamiento listo.")

**Selección Y Entrenamiento Del Modelo**

In [ ]:
# Estructura de la red neuronal convolucional (CNN)
model = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D(2, 2),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D(2, 2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes)
])

# Configurar el optimizador
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Entrenar el modelo
print("Iniciando entrenamiento...")
history = model.fit(train_data, train_labels, epochs=15, validation_data=(test_data, test_labels), verbose=1)

**Prueba Del Modelo**

In [ ]:
# Evaluar la exactitud (accuracy)
test_loss, test_acc = model.evaluate(test_data, test_labels, verbose=2)
print(f'\nExactitud en el conjunto de prueba: {test_acc:.4f} ({test_acc * 100:.2f}%)')

# Modelo con Softmax para obtener probabilidades
probability_model = keras.Sequential([
    model,
    keras.layers.Softmax()
])

# Realizar 5 predicciones con datos de prueba
predictions = probability_model.predict(test_data)

plt.figure(figsize=(12, 3))
print("\n*** Resultados de 5 predicciones ***")
for i in range(min(5, len(test_data))):
    # La predicción con mayor probabilidad
    predicted_idx = np.argmax(predictions[i])
    true_idx = test_labels[i]
    confidence = np.max(predictions[i]) * 100
    
    predicted_label = class_names[predicted_idx]
    true_label = class_names[true_idx]
    
    print(f"Predicción {i+1}: Valor Predicho = {predicted_label} ({confidence:.1f}%) | Valor Real = {true_label}")
    
    # Mostrar la imagen para verificar visualmente
    plt.subplot(1, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(test_data[i])
    plt.xlabel(f"P: {predicted_label}\nR: {true_label}")

plt.tight_layout()
plt.show()

**Guardar Modelo Entrenado (Para uso local con Cámara y Arduino)**

In [ ]:
# Guarda el modelo en Google Drive para descargarlo y usarlo en tu PC con la cámara y Arduino
model.save('/content/drive/MyDrive/modelo_colores_cnn.keras')
print("Modelo guardado exitosamente en Google Drive como 'modelo_colores_cnn.keras'.")